# Pylint Maintainability — Nesting Depth Raw Output Extraction

This notebook analyzes Python repositories with **Pylint** and captures the complete raw tool output for maintainability nesting-depth metric derivation and validation.

**Default benchmark repository:** [django/django](https://github.com/django/django)

The notebook supports:
- **Mode 1:** Clone from a Git repository URL
- **Mode 2:** Analyze an already-cloned local repository path

All deliverables are written to the configured `OUTPUT_DIR`.

## Section 1 — Install Dependencies

Install open-source packages required for repository acquisition, static analysis, and result processing.

In [ ]:
!pip install -q pylint pandas gitpython jupyter

## Section 2 — Configuration

Set execution mode, repository source, output location, and optional Pylint configuration file.

- Set `USE_GIT_URL = True` to clone from `REPO_URL`.
- Set `USE_GIT_URL = False` to analyze `LOCAL_REPO_PATH` directly.
- When cloning, use `IF_CLONE_EXISTS` to choose between reusing or re-cloning an existing local copy.

In [ ]:
USE_GIT_URL = False

REPO_URL = "https://github.com/django/django.git"

# Cross-platform path to the prepared repository clone
LOCAL_REPO_PATH = "./workspace/django"

OUTPUT_DIR = "./outputs"

PYLINT_RCFILE = None

# Clone behavior when USE_GIT_URL is True and the target directory already exists.
# Options: "reuse" (keep existing clone) or "reclone" (delete and clone again)
IF_CLONE_EXISTS = "reuse"

# Shallow clone depth for Git URL mode (None = full clone)
CLONE_DEPTH = 1

# Directory for cloned repositories (Git URL mode)
WORKSPACE_DIR = "./workspace"

# Pylint execution strategy:
#   "repo"      -> pylint <repo_path>  (preserves exact tool output)
#   "per_file"  -> pylint each .py file (continues on individual failures)
PYLINT_EXECUTION_MODE = "repo"

# Stream raw Pylint stdout/stderr to the notebook console during execution
STREAM_RAW_OUTPUT = True

# Number of raw output lines to preview in the notebook after execution
RAW_OUTPUT_PREVIEW_LINES = 150

# Optional cap for per-file mode only (None = analyze all files)
MAX_FILES = None

## Section 3 — Imports and Utility Functions

Modular helpers for logging, repository setup, file discovery, Pylint execution, and output extraction.

In [ ]:
from __future__ import annotations

import json
import os
import re
import shutil
import subprocess
import sys
import threading
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display
from git import Repo
from git.exc import GitCommandError, InvalidGitRepositoryError


EXCLUDED_DIR_NAMES = {
    ".git",
    "venv",
    ".venv",
    "env",
    "__pycache__",
    "build",
    "dist",
    "node_modules",
    ".tox",
    "migrations",
}

NESTING_DEPTH_SYMBOLS = {
    "too-many-nested-blocks",
    "old-too-many-nested-blocks",
}

NESTING_DEPTH_MESSAGE_IDS = {"R1702", "R0101"}

NESTING_KEYWORD_PATTERN = re.compile(
    r"too-many-nested-blocks|too many nested blocks|nested blocks",
    re.IGNORECASE,
)

NESTING_DEPTH_VALUE_PATTERN = re.compile(
    r"Too many nested blocks\s*\((\d+)\s*/\s*(\d+)\)",
    re.IGNORECASE,
)

PYLINT_LINE_PATTERN = re.compile(
    r"^(?P<file>[^:]+):(?P<line>\d+):(?P<column>\d+):\s*"
    r"(?P<type>\w+):\s*(?P<message>.+?)\s*\((?P<message_id>[\w-]+)\)\s*$"
)


class NotebookLogger:
    """Append-only logger for console progress and persistent error logging."""

    def __init__(self, error_log_path: Path) -> None:
        self.error_log_path = error_log_path
        self.error_log_path.parent.mkdir(parents=True, exist_ok=True)
        if not self.error_log_path.exists():
            self.error_log_path.write_text("", encoding="utf-8")

    def info(self, message: str) -> None:
        timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        print(f"[{timestamp}] INFO: {message}")

    def error(self, message: str) -> None:
        timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        line = f"[{timestamp}] ERROR: {message}\n"
        print(line, end="")
        with self.error_log_path.open("a", encoding="utf-8") as handle:
            handle.write(line)


def ensure_output_dir(output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)


def derive_clone_path(repo_url: str, workspace_dir: Path) -> Path:
    repo_name = repo_url.rstrip("/").removesuffix(".git").split("/")[-1]
    if not repo_name:
        raise ValueError(f"Unable to derive repository name from URL: {repo_url}")
    return workspace_dir / repo_name


def validate_repo_url(repo_url: str) -> None:
    if not repo_url or not isinstance(repo_url, str):
        raise ValueError("REPO_URL must be a non-empty string.")
    if not (repo_url.startswith("https://") or repo_url.startswith("git@") or repo_url.startswith("http://")):
        raise ValueError(f"Invalid repository URL format: {repo_url}")


def clone_or_reuse_repository(
    repo_url: str,
    workspace_dir: Path,
    if_clone_exists: str,
    logger: NotebookLogger,
    clone_depth: int | None = None,
) -> Path:
    validate_repo_url(repo_url)
    workspace_dir.mkdir(parents=True, exist_ok=True)
    clone_path = derive_clone_path(repo_url, workspace_dir)

    if clone_path.exists():
        if if_clone_exists == "reclone":
            logger.info(f"Removing existing clone at {clone_path}")
            shutil.rmtree(clone_path)
        elif if_clone_exists == "reuse":
            logger.info(f"Reusing existing clone at {clone_path}")
            return clone_path.resolve()
        else:
            raise ValueError('IF_CLONE_EXISTS must be either "reuse" or "reclone"')

    logger.info(f"Cloning {repo_url} into {clone_path} (depth={clone_depth})")
    try:
        clone_kwargs = {"depth": clone_depth} if clone_depth else {}
        Repo.clone_from(repo_url, clone_path, **clone_kwargs)
    except GitCommandError as exc:
        logger.error(f"Clone failed for {repo_url}: {exc}")
        raise

    logger.info(f"Clone completed: {clone_path}")
    return clone_path.resolve()


def validate_local_repository(local_repo_path: Path, logger: NotebookLogger) -> Path:
    if not local_repo_path.exists():
        message = f"Local repository path does not exist: {local_repo_path}"
        logger.error(message)
        raise FileNotFoundError(message)

    if not local_repo_path.is_dir():
        message = f"Local repository path is not a directory: {local_repo_path}"
        logger.error(message)
        raise NotADirectoryError(message)

    has_git = (local_repo_path / ".git").exists()
    has_python = any(local_repo_path.rglob("*.py"))

    if not has_git and not has_python:
        message = (
            f"Path is neither a Git repository nor a Python source directory: {local_repo_path}"
        )
        logger.error(message)
        raise ValueError(message)

    if has_git:
        try:
            Repo(local_repo_path)
        except InvalidGitRepositoryError as exc:
            logger.error(f"Invalid Git repository at {local_repo_path}: {exc}")
            raise

    logger.info(f"Validated local repository path: {local_repo_path}")
    return local_repo_path.resolve()


def resolve_repository_path(
    use_git_url: bool,
    repo_url: str,
    local_repo_path: str,
    workspace_dir: Path,
    if_clone_exists: str,
    logger: NotebookLogger,
    clone_depth: int | None = None,
) -> Path:
    if use_git_url:
        logger.info("Execution mode: Git repository URL")
        return clone_or_reuse_repository(
            repo_url, workspace_dir, if_clone_exists, logger, clone_depth=clone_depth
        )

    logger.info("Execution mode: Local repository path")
    return validate_local_repository(Path(local_repo_path), logger)


def configure_pythonpath(repo_path: Path) -> str:
    entries = [str(repo_path.resolve())]
    django_pkg = repo_path / "django"
    if django_pkg.is_dir() and (django_pkg / "__init__.py").exists():
        entries.append(str(django_pkg.resolve()))
    pythonpath = os.pathsep.join(dict.fromkeys(entries))
    os.environ["PYTHONPATH"] = pythonpath
    return pythonpath


def detect_pylint_rcfile(repo_path: Path) -> str | None:
    candidates = [
        repo_path / "pylintrc",
        repo_path / ".pylintrc",
        repo_path / "setup.cfg",
        repo_path / "pyproject.toml",
        repo_path / "tox.ini",
    ]
    for candidate in candidates:
        if not candidate.exists():
            continue
        if candidate.suffix == ".toml":
            text = candidate.read_text(encoding="utf-8", errors="replace")
            if "[tool.pylint" in text:
                return str(candidate)
        elif candidate.name in {"pylintrc", ".pylintrc"}:
            return str(candidate)
        else:
            text = candidate.read_text(encoding="utf-8", errors="replace")
            if "[pylint" in text.lower() or "[MASTER]" in text:
                return str(candidate)
    return None


def prepare_repository_for_pylint(
    repo_path: Path,
    output_path: Path,
    logger: NotebookLogger,
    rcfile_override: str | None = None,
) -> dict[str, Any]:
    repo_path = repo_path.resolve()
    pythonpath = configure_pythonpath(repo_path)
    python_files = discover_python_files(repo_path)
    stats = compute_repository_stats(repo_path, python_files)
    rcfile = rcfile_override or detect_pylint_rcfile(repo_path)

    git_head = None
    git_remote = None
    if (repo_path / ".git").exists():
        git_repo = Repo(repo_path)
        git_head = git_repo.head.commit.hexsha[:12]
        try:
            git_remote = git_repo.remotes.origin.url
        except Exception:
            git_remote = None

    manifest = {
        "prepared_at_utc": datetime.now(timezone.utc).isoformat(),
        "repo_path": str(repo_path),
        "git_head": git_head,
        "git_remote": git_remote,
        "python_file_count": stats["python_file_count"],
        "repository_size_bytes": stats["repository_size_bytes"],
        "directory_count": stats["directory_count"],
        "pythonpath": pythonpath,
        "pylint_rcfile": rcfile,
        "ready_for_pylint": stats["python_file_count"] > 0,
    }

    manifest_path = output_path / "repo_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    logger.info(f"Repository manifest saved: {manifest_path}")
    logger.info(f"PYTHONPATH configured: {pythonpath}")
    if rcfile:
        logger.info(f"Using Pylint rcfile: {rcfile}")
    else:
        logger.info("No repository Pylint rcfile detected; using Pylint defaults")

    return manifest


def should_exclude_path(path: Path) -> bool:
    return any(part in EXCLUDED_DIR_NAMES for part in path.parts)


def discover_python_files(repo_path: Path) -> list[Path]:
    python_files: list[Path] = []
    for file_path in repo_path.rglob("*.py"):
        if should_exclude_path(file_path.relative_to(repo_path)):
            continue
        python_files.append(file_path.resolve())
    python_files.sort()
    return python_files


def compute_repository_stats(repo_path: Path, python_files: list[Path]) -> dict[str, Any]:
    total_size_bytes = sum(file_path.stat().st_size for file_path in python_files)
    directory_count = sum(1 for current_path, dirnames, _ in os.walk(repo_path) if not should_exclude_path(current_path.relative_to(repo_path)))
    return {
        "python_file_count": len(python_files),
        "repository_size_bytes": total_size_bytes,
        "directory_count": directory_count,
    }


def save_python_file_list(python_files: list[Path], repo_path: Path, output_csv: Path) -> None:
    rows = [
        {
            "absolute_path": str(file_path),
            "relative_path": str(file_path.relative_to(repo_path)),
        }
        for file_path in python_files
    ]
    pd.DataFrame(rows).to_csv(output_csv, index=False)


def build_pylint_command(
    targets: Path | list[Path],
    output_format: str | None = None,
    rcfile: str | None = None,
) -> list[str]:
    target_list = [targets] if isinstance(targets, Path) else list(targets)
    command = [sys.executable, "-m", "pylint", *[str(target) for target in target_list]]
    command.extend(["--ignore=venv,.venv,env,build,dist,.tox,node_modules"])
    if output_format:
        command.extend(["--output-format", output_format])
    if rcfile:
        command.extend(["--rcfile", rcfile])
    return command


def run_pylint_command(
    command: list[str],
    logger: NotebookLogger,
    stream_raw: bool = False,
) -> tuple[str, str, int, bool]:
    try:
        if stream_raw:
            process = subprocess.Popen(
                command,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                encoding="utf-8",
                errors="replace",
                env=os.environ.copy(),
            )
            stdout_lines: list[str] = []
            stderr_lines: list[str] = []

            def _read_stream(pipe, sink, label):
                for line in iter(pipe.readline, ""):
                    print(line, end="", file=sys.stderr if label == "stderr" else sys.stdout)
                    sink.append(line)
                pipe.close()

            assert process.stdout is not None
            assert process.stderr is not None
            stdout_thread = threading.Thread(
                target=_read_stream, args=(process.stdout, stdout_lines, "stdout"), daemon=True
            )
            stderr_thread = threading.Thread(
                target=_read_stream, args=(process.stderr, stderr_lines, "stderr"), daemon=True
            )
            stdout_thread.start()
            stderr_thread.start()
            return_code = process.wait()
            stdout_thread.join()
            stderr_thread.join()
            stdout = "".join(stdout_lines)
            stderr = "".join(stderr_lines)
        else:
            completed = subprocess.run(
                command,
                capture_output=True,
                text=True,
                encoding="utf-8",
                errors="replace",
                check=False,
                env=os.environ.copy(),
            )
            stdout = completed.stdout
            stderr = completed.stderr
            return_code = completed.returncode

        success = return_code in (0, 1, 2, 4, 8, 16, 32)
        if not success:
            logger.error(
                f"Pylint command failed ({' '.join(command[2:4])}...) "
                f"with exit code {return_code}"
            )
        return stdout, stderr, return_code, success
    except Exception as exc:
        logger.error(f"Pylint execution exception: {exc}")
        return "", str(exc), -1, False


def combine_raw_streams(stdout: str, stderr: str) -> str:
    """Preserve Pylint stdout and stderr exactly as emitted."""
    raw_output = stdout
    if stderr:
        if raw_output and not raw_output.endswith("\n"):
            raw_output += "\n"
        raw_output += stderr
        if not raw_output.endswith("\n"):
            raw_output += "\n"
    return raw_output


def run_pylint_on_repo(
    repo_path: Path,
    rcfile: str | None,
    logger: NotebookLogger,
    stream_raw: bool = False,
) -> tuple[str, str, int, int]:
    logger.info(f"Running Pylint on repository path: {repo_path}")

    raw_command = build_pylint_command(repo_path, rcfile=rcfile)
    raw_stdout, raw_stderr, _, raw_success = run_pylint_command(
        raw_command, logger, stream_raw=stream_raw
    )
    raw_text = combine_raw_streams(raw_stdout, raw_stderr)

    json_command = build_pylint_command(repo_path, output_format="json", rcfile=rcfile)
    json_stdout, json_stderr, _, json_success = run_pylint_command(
        json_command, logger, stream_raw=False
    )
    if json_stderr.strip():
        logger.error(f"Pylint JSON stderr: {json_stderr.strip()}")

    if raw_success or json_success:
        return raw_text, json_stdout, 1, 0
    return raw_text, json_stdout, 0, 1


def run_pylint_on_file(
    python_file: Path,
    output_format: str | None,
    rcfile: str | None,
    logger: NotebookLogger,
) -> tuple[str, str, int, bool]:
    command = build_pylint_command(python_file, output_format=output_format, rcfile=rcfile)
    stdout, stderr, return_code, success = run_pylint_command(
        command, logger, stream_raw=False
    )
    return stdout, stderr, return_code, success


def run_pylint_batch(
    python_files: list[Path],
    rcfile: str | None,
    logger: NotebookLogger,
    stream_raw: bool = False,
) -> tuple[str, str, int, int]:
    raw_chunks: list[str] = []
    json_chunks: list[str] = []
    success_count = 0
    failure_count = 0

    total_files = len(python_files)
    for index, python_file in enumerate(python_files, start=1):
        if index == 1 or index % 25 == 0 or index == total_files:
            logger.info(f"Running Pylint ({index}/{total_files}): {python_file.name}")

        raw_stdout, raw_stderr, _, raw_success = run_pylint_on_file(
            python_file, output_format=None, rcfile=rcfile, logger=logger
        )
        if stream_raw:
            if raw_stdout:
                print(raw_stdout, end="")
            if raw_stderr:
                print(raw_stderr, end="", file=sys.stderr)

        json_stdout, json_stderr, _, json_success = run_pylint_on_file(
            python_file, output_format="json", rcfile=rcfile, logger=logger
        )

        raw_chunks.append(combine_raw_streams(raw_stdout, raw_stderr))

        if json_stdout.strip():
            json_chunks.append(json_stdout.strip())
        if json_stderr.strip():
            logger.error(f"Pylint JSON stderr for {python_file}: {json_stderr.strip()}")

        if raw_success or json_success:
            success_count += 1
        else:
            failure_count += 1

    raw_text = "".join(raw_chunks)
    json_text = "\n".join(chunk for chunk in json_chunks if chunk)
    return raw_text, json_text, success_count, failure_count


def merge_pylint_json_outputs(json_text: str, logger: NotebookLogger) -> list[dict[str, Any]]:
    if not json_text.strip():
        return []
    try:
        parsed = json.loads(json_text)
    except json.JSONDecodeError as exc:
        logger.error(f"Failed to parse Pylint JSON output: {exc}")
        return []
    if isinstance(parsed, list):
        return parsed
    logger.error("Unexpected Pylint JSON payload; expected a list.")
    return []


def preview_raw_output(raw_text: str, preview_lines: int, output_path: Path) -> None:
    lines = raw_text.splitlines()
    print(f"\n{'=' * 80}")
    print(f"RAW PYLINT OUTPUT PREVIEW (first {preview_lines} lines)")
    print(f"{'=' * 80}\n")
    if not lines:
        print("(empty raw output)")
        return
    print("\n".join(lines[:preview_lines]))
    remaining = len(lines) - preview_lines
    if remaining > 0:
        print(f"\n... ({remaining} more lines saved to {output_path})")


def pylint_json_to_dataframe(records: list[dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for record in records:
        rows.append(
            {
                "file": record.get("path") or record.get("module", ""),
                "line": record.get("line", ""),
                "column": record.get("column", ""),
                "type": record.get("type", ""),
                "symbol": record.get("symbol", ""),
                "message": record.get("message", ""),
                "message-id": record.get("message-id", ""),
                "confidence": record.get("confidence", ""),
            }
        )
    columns = [
        "file",
        "line",
        "column",
        "type",
        "symbol",
        "message",
        "message-id",
        "confidence",
    ]
    return pd.DataFrame(rows, columns=columns)


def extract_detected_nesting_depth(message: str) -> int | None:
    match = NESTING_DEPTH_VALUE_PATTERN.search(message)
    if match:
        return int(match.group(1))
    return None


def is_nesting_depth_finding(record: dict[str, Any]) -> bool:
    symbol = str(record.get("symbol", "")).lower()
    message_id = str(record.get("message-id", "")).upper()
    message = str(record.get("message", ""))
    return (
        symbol in NESTING_DEPTH_SYMBOLS
        or message_id in NESTING_DEPTH_MESSAGE_IDS
        or bool(NESTING_KEYWORD_PATTERN.search(message))
    )


def extract_nesting_findings_from_json(records: list[dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for record in records:
        if not is_nesting_depth_finding(record):
            continue
        message = str(record.get("message", ""))
        rows.append(
            {
                "file": record.get("path") or record.get("module", ""),
                "line": record.get("line", ""),
                "warning": record.get("symbol", ""),
                "message": message,
                "detected_nesting_depth": extract_detected_nesting_depth(message),
            }
        )
    return pd.DataFrame(
        rows,
        columns=["file", "line", "warning", "message", "detected_nesting_depth"],
    )


def extract_nesting_findings_from_raw(raw_text: str) -> pd.DataFrame:
    rows = []
    for line in raw_text.splitlines():
        if not NESTING_KEYWORD_PATTERN.search(line):
            continue
        match = PYLINT_LINE_PATTERN.match(line.strip())
        if match:
            message = match.group("message")
            rows.append(
                {
                    "file": match.group("file"),
                    "line": int(match.group("line")),
                    "warning": match.group("message_id"),
                    "message": message,
                    "detected_nesting_depth": extract_detected_nesting_depth(message),
                }
            )
        else:
            rows.append(
                {
                    "file": "",
                    "line": "",
                    "warning": "nested-blocks-warning",
                    "message": line.strip(),
                    "detected_nesting_depth": extract_detected_nesting_depth(line),
                }
            )
    return pd.DataFrame(
        rows,
        columns=["file", "line", "warning", "message", "detected_nesting_depth"],
    )


def combine_nesting_findings(json_df: pd.DataFrame, raw_df: pd.DataFrame) -> pd.DataFrame:
    combined = pd.concat([json_df, raw_df], ignore_index=True)
    if combined.empty:
        return combined
    dedupe_columns = ["file", "line", "warning", "message"]
    return combined.drop_duplicates(subset=dedupe_columns, keep="first").reset_index(drop=True)

## Section 3 — Repository Setup

Resolve the repository path based on configuration, initialize output directories, and display repository statistics.

In [ ]:
OUTPUT_PATH = Path(OUTPUT_DIR).resolve()
WORKSPACE_PATH = Path(WORKSPACE_DIR).resolve()
ERROR_LOG_PATH = OUTPUT_PATH / "error_log.txt"

ensure_output_dir(OUTPUT_PATH)
logger = NotebookLogger(ERROR_LOG_PATH)

try:
    REPO_PATH = resolve_repository_path(
        use_git_url=USE_GIT_URL,
        repo_url=REPO_URL,
        local_repo_path=LOCAL_REPO_PATH,
        workspace_dir=WORKSPACE_PATH,
        if_clone_exists=IF_CLONE_EXISTS,
        logger=logger,
    )
except Exception as exc:
    logger.error(f"Repository setup failed: {exc}")
    raise

logger.info(f"Repository ready at: {REPO_PATH}")

## Section 5 — Discover Python Files

Recursively discover `.py` files while excluding common non-source directories (including `migrations`).

In [ ]:
PYTHON_FILES = discover_python_files(REPO_PATH)
REPO_STATS = compute_repository_stats(REPO_PATH, PYTHON_FILES)

PYTHON_FILES_CSV = OUTPUT_PATH / "python_files.csv"
save_python_file_list(PYTHON_FILES, REPO_PATH, PYTHON_FILES_CSV)

print(f"Total Python Files Found: {len(PYTHON_FILES)}")
print(f"Repository Size (Python files only): {REPO_STATS['repository_size_bytes']:,} bytes")
print(f"Total Directories (excluding filtered paths): {REPO_STATS['directory_count']:,}")
print(f"Saved file list to: {PYTHON_FILES_CSV}")

## Section 5 — Execute Pylint

Run Pylint against each discovered Python file. Execution continues even when individual files fail. Raw stdout and stderr are preserved without suppression.

In [ ]:
if not PYTHON_FILES:
    logger.error("No Python files discovered; skipping Pylint execution.")
    raw_text_output = ""
    json_text_output = ""
    FILES_SUCCESS = 0
    FILES_FAILED = 0
else:
    analysis_files = PYTHON_FILES
    if MAX_FILES is not None:
        analysis_files = PYTHON_FILES[:MAX_FILES]
        logger.info(f"MAX_FILES enabled: analyzing first {len(analysis_files)} files")

    if PYLINT_EXECUTION_MODE == "repo":
        raw_text_output, json_text_output, FILES_SUCCESS, FILES_FAILED = run_pylint_on_repo(
            repo_path=REPO_PATH,
            rcfile=EFFECTIVE_PYLINT_RCFILE,
            logger=logger,
            stream_raw=STREAM_RAW_OUTPUT,
        )
        if FILES_SUCCESS:
            FILES_SUCCESS = len(PYTHON_FILES)
            FILES_FAILED = 0
        else:
            FILES_SUCCESS = 0
            FILES_FAILED = len(PYTHON_FILES)
    else:
        raw_text_output, json_text_output, FILES_SUCCESS, FILES_FAILED = run_pylint_batch(
            python_files=analysis_files,
            rcfile=PYLINT_RCFILE,
            logger=logger,
            stream_raw=STREAM_RAW_OUTPUT,
        )

logger.info(
    f"Pylint execution complete. Success: {FILES_SUCCESS}, Failed: {FILES_FAILED}"
)
logger.info(f"Raw output size: {len(raw_text_output):,} characters")

## Section 6 — Raw Output Extraction

Persist complete raw Pylint text output, structured JSON output, and a CSV representation of all findings.

In [ ]:
RAW_OUTPUT_PATH = OUTPUT_PATH / "pylint_raw_output.txt"
JSON_OUTPUT_PATH = OUTPUT_PATH / "pylint_output.json"
RESULTS_CSV_PATH = OUTPUT_PATH / "pylint_results.csv"

raw_text_output = "".join(RAW_CHUNKS)
RAW_OUTPUT_PATH.write_text(raw_text_output, encoding="utf-8")

PYLINT_JSON_RECORDS = merge_pylint_json_outputs(JSON_CHUNKS, logger)
JSON_OUTPUT_PATH.write_text(
    json.dumps(PYLINT_JSON_RECORDS, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

PYLINT_RESULTS_DF = pylint_json_to_dataframe(PYLINT_JSON_RECORDS)
PYLINT_RESULTS_DF.to_csv(RESULTS_CSV_PATH, index=False)

logger.info(f"Saved raw output: {RAW_OUTPUT_PATH}")
logger.info(f"Saved JSON output: {JSON_OUTPUT_PATH}")
logger.info(f"Saved CSV results: {RESULTS_CSV_PATH}")
logger.info(f"Total Pylint findings: {len(PYLINT_RESULTS_DF)}")

## Section 8 — Nesting Depth Evidence Extraction

Extract maintainability nesting-depth evidence from both structured JSON and raw Pylint output.

Target rules include:
- `too-many-nested-blocks` (`R1702`)
- `old-too-many-nested-blocks` (`R0101`)
- Additional nested-block warnings present in raw output

In [ ]:
NESTING_JSON_DF = extract_nesting_findings_from_json(PYLINT_JSON_RECORDS)
NESTING_RAW_DF = extract_nesting_findings_from_raw(raw_text_output)
NESTING_FINDINGS_DF = combine_nesting_findings(NESTING_JSON_DF, NESTING_RAW_DF)

NESTING_FINDINGS_CSV = OUTPUT_PATH / "nesting_depth_findings.csv"
NESTING_FINDINGS_DF.to_csv(NESTING_FINDINGS_CSV, index=False)

logger.info(f"Saved nesting depth findings: {NESTING_FINDINGS_CSV}")
logger.info(f"Nesting depth findings count: {len(NESTING_FINDINGS_DF)}")

if not NESTING_FINDINGS_DF.empty:
    display(NESTING_FINDINGS_DF.head(10))
else:
    print("No nesting depth findings detected.")

## Section 9 — Summary Dashboard

Overview of analysis coverage, Pylint findings, and nesting-depth evidence.

In [ ]:
summary_df = pd.DataFrame(
    [
        {"Metric": "Total Python Files", "Value": len(PYTHON_FILES)},
        {"Metric": "Files Successfully Analyzed", "Value": FILES_SUCCESS},
        {"Metric": "Files Failed", "Value": FILES_FAILED},
        {"Metric": "Total Pylint Findings", "Value": len(PYLINT_RESULTS_DF)},
        {"Metric": "Nesting Depth Findings", "Value": len(NESTING_FINDINGS_DF)},
    ]
)

display(summary_df)

deliverables = [
    RAW_OUTPUT_PATH,
    JSON_OUTPUT_PATH,
    RESULTS_CSV_PATH,
    PYTHON_FILES_CSV,
    NESTING_FINDINGS_CSV,
    ERROR_LOG_PATH,
]

print("\nDeliverables:")
for deliverable in deliverables:
    status = "OK" if deliverable.exists() else "MISSING"
    print(f"  [{status}] {deliverable}")

## Section 10 — Error Handling Notes

Failures encountered during cloning, validation, or Pylint execution are appended to `outputs/error_log.txt`.

Review the log below if any step reported errors.

In [ ]:
if ERROR_LOG_PATH.exists() and ERROR_LOG_PATH.stat().st_size > 0:
    print(ERROR_LOG_PATH.read_text(encoding="utf-8"))
else:
    print("No errors logged.")

## Section 10 — Deliverables

Upon successful completion, the following artifacts are available under `outputs/`:

```text
outputs/
├── pylint_raw_output.txt
├── pylint_output.json
├── pylint_results.csv
├── python_files.csv
├── nesting_depth_findings.csv
└── error_log.txt
```

The notebook is designed to run end-to-end in Jupyter Notebook and Google Colab without manual intervention.